# Simple CNN for Traffic Density Classification

Mục tiêu của notebook là xây dựng một mô hình CNN cơ bản để phân loại ảnh giao thông thành 5 mức mật độ: `Empty`, `Low`, `Medium`, `High` và `Traffic Jam`.

In [ ]:
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
project_dir = Path.cwd()
if not (project_dir / "src").exists():
    project_dir = project_dir.parent

if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src.data_loader import CLASS_NAMES, create_generators
from src.evaluation import evaluate_model

## 1. Thiết lập thí nghiệm

In [ ]:
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
USE_AUGMENTATION = False

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

`USE_AUGMENTATION=False` được dùng cho mô hình baseline. Có thể đổi thành `True` để chạy lại cùng kiến trúc với augmentation.

## 2. Đọc dữ liệu

In [ ]:
train_generator, val_generator, test_generator = create_generators(
    augmentation=USE_AUGMENTATION,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

In [ ]:
print("Class mapping:", train_generator.class_indices)
print("Training images:", train_generator.samples)
print("Validation images:", val_generator.samples)
print("Test images:", test_generator.samples)

## 3. Hiển thị một batch ảnh

In [ ]:
images, labels = next(train_generator)

plt.figure(figsize=(12, 8))

for index in range(12):
    plt.subplot(3, 4, index + 1)
    plt.imshow(images[index])
    plt.title(CLASS_NAMES[np.argmax(labels[index])])
    plt.axis("off")

plt.tight_layout()
plt.show()

train_generator.reset()

## 4. Kiến trúc Simple CNN

Mô hình gồm ba tầng convolution để trích xuất đặc trưng, ba tầng max pooling để giảm kích thước đặc trưng và hai tầng fully connected để phân loại.

```text
Input 224×224×3
        ↓
Conv2D: 16 filters, 3×3, ReLU
        ↓
MaxPooling2D: 2×2
        ↓
Conv2D: 32 filters, 3×3, ReLU
        ↓
MaxPooling2D: 2×2
        ↓
Conv2D: 64 filters, 3×3, ReLU
        ↓
MaxPooling2D: 2×2
        ↓
Flatten → Dense 64, ReLU
        ↓
Dense 5, Softmax
```

In [ ]:
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Conv2D(16, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(5, activation="softmax"),
])

model.summary()

- `Conv2D` học các đặc trưng như cạnh, hình dạng xe và vùng đường.
- `MaxPooling2D` giảm kích thước feature map và chi phí tính toán.
- `Flatten` chuyển feature map thành một vector.
- `Dense(64)` kết hợp các đặc trưng để phân loại.
- `Dense(5, softmax)` trả về xác suất của 5 lớp.

## 5. Compile model

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

## 6. Huấn luyện model

In [ ]:
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    verbose=1,
)

## 7. Training history

In [ ]:
epochs = range(1, len(history.history["loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history.history["loss"], label="Training loss")
plt.plot(epochs, history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history.history["accuracy"], label="Training accuracy")
plt.plot(epochs, history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## 8. Đánh giá trên test set

Precision, Recall và F1-score được tính theo macro average để mỗi lớp có mức đóng góp như nhau.

In [ ]:
model_name = "Simple CNN + Augmentation" if USE_AUGMENTATION else "Simple CNN"

result = evaluate_model(
    model=model,
    test_generator=test_generator,
    model_name=model_name,
)

result

## 9. Lưu model

In [ ]:
output_dir = project_dir / "outputs"
output_dir.mkdir(exist_ok=True)

model_path = output_dir / "simple_cnn.keras"
model.save(model_path)

print(f"Model saved to: {model_path}")